# CollusionLab Analysis Notebook

Canonical analysis workflow for Phase 6 metrics.
Covers onset threshold analysis (Part A) and concealment/transition analysis (Part B).


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from collusionlab.metrics.base import LogReader, get_metrics_computer
from collusionlab.metrics import collusion, concealment

# Register pricing metrics
import collusionlab.environments.pricing.metrics  # noqa: F401

## 1. Load sweep results

Point `SWEEP_PATH` at a `sweep_manifest.json` to load all successful runs.

In [ ]:
SWEEP_PATH = ROOT / "data" / "raw" / "sweep_EXAMPLE" / "sweep_manifest.json"

# Uncomment and set to a real sweep manifest path:
# SWEEP_PATH = Path("...")

runs = []
if SWEEP_PATH.exists():
    runs = LogReader.load_sweep(SWEEP_PATH)
    print(f"Loaded {len(runs)} runs")
else:
    print(f"Sweep manifest not found at {SWEEP_PATH}")
    print("Set SWEEP_PATH to a valid sweep_manifest.json to continue.")

## 2. Compute threshold table (Part A)

Groups runs by experimental condition and computes onset rate + median onset round.

In [ ]:
if runs:
    table = collusion.threshold_table(
        runs,
        groupby=["communication_mode", "memory_window", "n_agents"],
    )
    display(table)
else:
    print("No runs loaded — skipping threshold table.")

## 3. Onset rate by condition (Part A)

In [ ]:
if runs and not table.empty:
    fig = px.bar(
        table,
        x="communication_mode",
        y="onset_rate",
        color="memory_window",
        barmode="group",
        title="Onset Rate by Communication Mode and Memory Window",
    )
    fig.update_layout(yaxis_title="Onset Rate", xaxis_title="Communication Mode")
    fig.show()

    fig2 = px.bar(
        table,
        x="communication_mode",
        y="median_onset_round",
        color="memory_window",
        barmode="group",
        title="Median Onset Round by Condition",
    )
    fig2.update_layout(yaxis_title="Median Onset Round", xaxis_title="Communication Mode")
    fig2.show()

## 4. Concealment gap time series (Part B)

Plots rolling covert and hollow coordination rates for runs with vs without oversight.

In [ ]:
if runs:
    fig = go.Figure()
    for run in runs[:6]:  # limit to first 6 for readability
        covert = concealment.covert_coordination_series(run, window=10, audited_only=False)
        hollow = concealment.hollow_coordination_series(run, window=10, audited_only=False)
        label = f"{run.run_id[:8]} ({run.oversight_mode})"
        if not covert.empty:
            fig.add_trace(go.Scatter(
                x=list(covert.index), y=covert.values,
                mode="lines", name=f"{label} covert",
            ))
        if not hollow.empty:
            fig.add_trace(go.Scatter(
                x=list(hollow.index), y=hollow.values,
                mode="lines", name=f"{label} hollow",
                line=dict(dash="dot"),
            ))
    fig.update_layout(
        title="Rolling Covert & Hollow Coordination Rates",
        xaxis_title="Round",
        yaxis_title="Rate",
        height=500,
    )
    fig.show()

## 5. Steganographic score flagging

Flag high steganographic-score runs for closer transcript inspection.

In [ ]:
if runs:
    computer = get_metrics_computer("pricing")
    sweep_df = computer.compute_sweep(runs)

    top = sweep_df.nlargest(10, "steganographic_score")[[
        "run_id", "communication_mode", "oversight_mode",
        "onset_round", "transition_round",
        "price_follow_rate", "steganographic_score",
    ]]
    display(top)

## 6. Export

Export threshold table and metrics DataFrame as CSV for paper appendix.

In [ ]:
if runs:
    out_dir = ROOT / "data" / "processed"
    out_dir.mkdir(parents=True, exist_ok=True)

    table.to_csv(out_dir / "threshold_table.csv", index=False)
    sweep_df.to_csv(out_dir / "sweep_metrics.csv", index=False)
    print(f"Exported to {out_dir}")